### Imports

In [ ]:
import os
import torch
import torchvision.transforms as T
from datasets import load_dataset
from PIL import Image

## Setup
we want all our images to be 224 x 224 with 8 frames each. In other words our tensor would be something like 8 x 3 x 224 x 224.

In [ ]:
IMG_SIZE = 224
N_FRAMES = 8
SAVE_PATH = "../data/dataset"       # where to save the data after processing

In [ ]:
# function to transform images into multiple frames
def image_to_clip(img, transform, n):
    frames = [transform(img) for _ in range(n)]
    return torch.stack(frames)

# function to save the data
def save(data, folder, idx, prefix=""):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, f"{prefix}_{idx:06d}.pt")
    torch.save(data, path)

## UTKFace data (positive examples)

In [ ]:
# function to extract the age label
def parse_utkface_label(filename):
     # the first part of the name of the utkface data is the age
    return int(filename.split("_")[0])

# the augmentation transform
augment1 = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

### create samples
The UTKFace data were downloaded locally beforehand, so label extraction is done manually.

In [ ]:
utkface_path = "../data/UTKFace/"

for i, filename in enumerate(os.listdir(utkface_path)):
    img = Image.open(os.path.join(utkface_path, filename)).convert("RGB")
    age = parse_utkface_label(filename)
    clip = image_to_clip(img, augment1, N_FRAMES)

    sample = {
        "frames": clip,
        "age": age,
        "person": 1
    }

    save(sample, SAVE_PATH, prefix="pos", idx=i)

## Places365 data (negative examples)

In [ ]:
augment2 = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ColorJitter(0.4, 0.4, 0.4, 0.2),
    T.RandomGrayscale(p=0.3),
    T.GaussianBlur(3),
    T.ToTensor()
])

### create samples
Here hugging face is used to access the Places365 data. ``streaming=True`` is used so there won't be need to download the entire datset.

In [ ]:
places365 = load_dataset("Andron00e/Places365-custom", split="train", streaming=True)
n_examples = 2000

for i, data in enumerate(places365):
    if i >= n_examples:
        break

    img = data["image"].convert("RGB")
    clip = image_to_clip(img, augment2, N_FRAMES)

    sample = {
        "frames": clip,
        "age": -1,
        "person": 0
    }

    save(sample, SAVE_PATH, prefix="neg", idx=i)